# Notebook 2 — Django ORM Practice
## Learn how Django Python code talks to a database

**Purpose:** This notebook teaches ORM ideas using a small, self-contained Django example. It does not replace the main `django_lab` project.

**You will learn**
1. What ORM means.
2. How models represent database tables.
3. How to create and read objects.
4. How to filter, order and search.
5. How to update and delete.
6. How relationships work.
7. What QuerySets are.
8. How to think about database query efficiency.

**Important setup:** This notebook uses Django with SQLite only to keep the notebook exercise easy to start. The full Day 2 practical uses PostgreSQL. The same Django ORM concepts apply to both.

## 1. Prepare a small Django environment

Run this notebook using a Python environment where Django is installed. If Django is not installed in the notebook's environment, run this in a terminal:
```bash
python -m pip install "Django>=5.2,<6.0"
```

The next cell creates a small Django configuration inside this notebook process. You do not need to create a separate project for this exercise.

In [ ]:
import os
import django
from django.conf import settings

if not settings.configured:
    settings.configure(
        SECRET_KEY="not-a-real-secret-for-local-learning",
        INSTALLED_APPS=["django.contrib.contenttypes"],
        DATABASES={
            "default": {
                "ENGINE": "django.db.backends.sqlite3",
                "NAME": "django_orm_practice.sqlite3",
            }
        },
        DEFAULT_AUTO_FIELD="django.db.models.AutoField",
        USE_TZ=True,
    )

django.setup()
print("Django is ready:", django.get_version())

## 2. Define a model

A Django model is a Python class that describes data to be stored in a database.

We will create a tiny `Book` model. Each object represents one book record.

In [ ]:
from django.db import models, connection

class Book(models.Model):
    title = models.CharField(max_length=120)
    author = models.CharField(max_length=80)
    page_count = models.PositiveIntegerField(default=1)
    is_available = models.BooleanField(default=True)

    class Meta:
        app_label = "orm_practice"
        ordering = ["title"]

    def __str__(self):
        return self.title

print("Model class created:", Book.__name__)

### Understand each field
- `CharField`: stores short text.
- `max_length`: maximum number of characters.
- `PositiveIntegerField`: stores a non-negative whole number.
- `BooleanField`: stores True or False.
- `default`: value used when a new object does not provide one.
- `Meta.ordering`: default result order.
- `__str__`: readable object description.

A model is a structure. A row is one saved record.

## 3. Create the table for this notebook

In a normal Django project, you use `makemigrations` and `migrate`. In this compact notebook example, we create the table directly so the learner can focus on ORM queries.

Do not use `schema_editor` as a replacement for migrations in a real Django project.

In [ ]:
with connection.schema_editor() as schema_editor:
    try:
        schema_editor.delete_model(Book)
    except Exception:
        pass
    schema_editor.create_model(Book)

print("Book table is ready.")

## 4. Create objects

The ORM can create a row and return a Python object.

In [ ]:
book1 = Book.objects.create(
    title="Python for Beginners",
    author="Asha",
    page_count=180,
    is_available=True,
)

book2 = Book.objects.create(
    title="Learning Django",
    author="Rahul",
    page_count=240,
    is_available=True,
)

book3 = Book.objects.create(
    title="Database Basics",
    author="Asha",
    page_count=120,
    is_available=False,
)

print(book1.id, book1.title)
print(book2.id, book2.title)
print(book3.id, book3.title)

`objects` is the model's default manager. `create()` inserts a database row and returns the new model object.

The `id` is the primary key Django creates automatically.

## 5. Read records

`all()` returns a QuerySet representing all records.

In [ ]:
all_books = Book.objects.all()
list(all_books.values("id", "title", "author", "page_count", "is_available"))

A QuerySet is a Django object that represents a database query and its results. QuerySets are often lazy: Django waits until the data is needed before running the query.

`values()` returns dictionaries containing selected fields. This is useful for learning and inspection.

## 6. Filter records

`filter()` returns all records matching the condition. It can return zero, one or many results.

In [ ]:
available_books = Book.objects.filter(is_available=True)
list(available_books.values("title", "author"))

### Common lookup examples
- `title__icontains="django"`: title contains text, ignoring case.
- `page_count__gt=150`: page count greater than 150.
- `page_count__gte=150`: greater than or equal to 150.
- `author="Asha"`: exact match.

In [ ]:
django_books = Book.objects.filter(title__icontains="django")
long_books = Book.objects.filter(page_count__gt=150)
asha_books = Book.objects.filter(author="Asha")

list(django_books.values_list("title", flat=True)), \
list(long_books.values_list("title", flat=True)), \
list(asha_books.values_list("title", flat=True))

## 7. Get exactly one record

`get()` is for a query expected to return exactly one object. It raises `DoesNotExist` if no row matches and `MultipleObjectsReturned` if more than one row matches.

For website views, developers often use `get_object_or_404()` so a missing record becomes a 404 response.

In [ ]:
one_book = Book.objects.get(title="Learning Django")
one_book.title, one_book.author

## 8. Sort results

`order_by()` sorts the results. A minus sign means descending order.

In [ ]:
list(Book.objects.order_by("-page_count").values_list("title", "page_count"))

## 9. Update a record

Change the Python object's value and call `save()`. The save writes the change to the database.

In [ ]:
one_book.page_count = 260
one_book.save()

Book.objects.get(pk=one_book.pk).page_count

You can also update a QuerySet directly:
```python
Book.objects.filter(author="Asha").update(is_available=True)
```
This runs a database update directly. It does not call each object's `save()` method. Understand this difference before using `update()` in code that depends on custom save logic.

## 10. Delete a record

Deleting is permanent unless you have a backup or transaction that can restore the data. Practise only with sample records.

In [ ]:
book_to_remove = Book.objects.get(title="Database Basics")
deleted_count, details = book_to_remove.delete()
deleted_count, details

## 11. Understand relationships

In the main Django practical:
- A user has one profile: OneToOneField.
- A user can write many articles: ForeignKey on Article.
- An article can have many categories and each category can be used by many articles: ManyToManyField.

This notebook's small model has no relationships, so here is the shape of a ForeignKey model:
```python
class Writer(models.Model):
    name = models.CharField(max_length=80)

class Story(models.Model):
    title = models.CharField(max_length=120)
    writer = models.ForeignKey(
        Writer,
        on_delete=models.PROTECT,
        related_name="stories"
    )
```
One Writer can be connected to many Story rows. Each Story points to one Writer.

## 12. Query efficiency

Suppose a page displays 50 articles and each article's author name. A poorly designed query pattern may load the article list once and then query the author separately for every article. This is called the N+1 query problem.

For a ForeignKey or OneToOne relation, `select_related()` can fetch related data with a SQL JOIN.

For ManyToMany or reverse relations, `prefetch_related()` commonly fetches related rows in another query and joins them in Python.

Example from the main course:
```python
Article.objects.select_related("author")
Article.objects.prefetch_related("categories")
```

Use these when the page needs the related data. Do not add them blindly.

## 13. Look at the SQL behind a QuerySet

Django can show the SQL query represented by a QuerySet:

In [ ]:
queryset = Book.objects.filter(is_available=True).order_by("title")
print(queryset.query)

The exact SQL can vary by database engine. The important idea is that Django translates Python ORM instructions into SQL.

## 14. Exercises

1. Create a fourth book with your own title and author.
2. Find all books by Asha.
3. Find books with more than 200 pages.
4. Sort books by title in descending order.
5. Change one book's availability to False.
6. Delete only the fourth book.
7. Explain the difference between `filter()` and `get()`.
8. Explain why `select_related()` may reduce database queries.

In [ ]:
# Exercise answer examples — attempt the tasks before running this cell.

fourth_book = Book.objects.create(
    title="Practical Web Development",
    author="Meera",
    page_count=310,
    is_available=True,
)

answer_2 = list(Book.objects.filter(author="Asha").values_list("title", flat=True))
answer_3 = list(Book.objects.filter(page_count__gt=200).values_list("title", flat=True))
answer_4 = list(Book.objects.order_by("-title").values_list("title", flat=True))

fourth_book.is_available = False
fourth_book.save()

fourth_book.delete()

answer_2, answer_3, answer_4

## 15. Connect this notebook to the real Django practical

Now open the project in VS Code and inspect:
- `django_lab/journal/models.py` — the real Article and Category models.
- `django_lab/journal/views.py` — QuerySets used by website pages.
- `django_lab/journal/forms.py` — form validation.
- `django_lab/config/settings.py` — PostgreSQL connection settings.

The notebook used SQLite for a quick isolated exercise. The main application uses PostgreSQL and migrations. Follow the course's PostgreSQL setup guide before running the main application.

## 16. Key takeaways
- ORM connects Python model objects to relational database records.
- Models describe data structure.
- QuerySets represent database queries.
- `filter()` can return many records; `get()` expects exactly one.
- `save()` persists object changes.
- `delete()` removes a record.
- Relationships connect records in different tables.
- SQL understanding remains important even when using an ORM.